# Notebook 6 — Anomaly Detection: Fake/Suspicious Review Detection
### Isolation Forest · Autoencoder (Keras)

| Item | Detail |
|------|--------|
| **Input** | `gold_reviews.parquet` |
| **Task** | Unsupervised anomaly detection (no ground-truth labels) |
| **Model 1** | Isolation Forest — tabular features |
| **Model 2** | Autoencoder — reconstruction error on tabular features |
| **Output** | Anomaly scores + flagged reviews saved to `results/` |

> **Strategy:** We treat reviews with unusual statistical patterns as _potentially_ fake or suspicious.
> Both models are trained unsupervised — no ground-truth fraud labels are needed.
> Reviews flagged by **both** models are considered high-confidence suspicious cases.
>
> **Data Leakage Note:** `user_star_deviation` and `biz_star_deviation` are excluded from features
> because they are computed directly from `stars`, which would artificially inflate anomaly detection
> toward 1-star and 5-star reviews rather than genuinely suspicious patterns.

## 1 — Install Dependencies

In [1]:
!pip install pyarrow scikit-learn tensorflow matplotlib seaborn joblib -q

## 2 — Imports

In [2]:
import os, warnings, joblib
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('TensorFlow version :', tf.__version__)
print('NumPy version      :', np.__version__)
print('GPU devices        :', tf.config.list_physical_devices('GPU'))

TensorFlow version : 2.20.0
NumPy version      : 2.0.2
GPU devices        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4 — Paths & Directories

In [4]:
RESULTS_DIR = '/content/drive/MyDrive/Dataset/results'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
MODELS_DIR  = f'{RESULTS_DIR}/models'
PLOTS_DIR   = f'{RESULTS_DIR}/anomaly_plots'

GOLD_PARQUET = f'{GOLD_DIR}/gold_reviews.parquet'

for d in [MODELS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

def save_fig(name):
    path = os.path.join(PLOTS_DIR, f'{name}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Saved -> {path}')

print('Directories ready.')
print(f'  Gold parquet : {GOLD_PARQUET}')
print(f'  Models dir   : {MODELS_DIR}')
print(f'  Plots dir    : {PLOTS_DIR}')

Directories ready.
  Gold parquet : /content/drive/MyDrive/Dataset/results/gold/gold_reviews.parquet
  Models dir   : /content/drive/MyDrive/Dataset/results/models
  Plots dir    : /content/drive/MyDrive/Dataset/results/anomaly_plots


In [5]:
import shutil, os
import pyarrow.parquet as _pq

LOCAL_GOLD = '/content/gold_reviews.parquet'
GOLD_SRC   = f'{GOLD_DIR}/gold_reviews.parquet'

def _local_cache_valid(path):
    """Return True only if path contains at least one parquet file with a valid footer."""
    if not os.path.isdir(path):
        return False
    parquet_files = [f for f in os.listdir(path) if f.endswith('.parquet')]
    if not parquet_files:
        return False
    try:
        _pq.read_metadata(os.path.join(path, parquet_files[0]))
        return True
    except Exception:
        return False

if _local_cache_valid(LOCAL_GOLD):
    print('Local gold parquet already cached and valid.')
else:
    # Remove corrupt or incomplete cache before re-copying.
    if os.path.exists(LOCAL_GOLD):
        shutil.rmtree(LOCAL_GOLD)
        print('Removed corrupt/incomplete local cache — will re-copy.')
    os.makedirs(LOCAL_GOLD, exist_ok=True)
    print('Copying gold parquet files to local disk ...')

    # Skip hidden .crc sidecar files — they trigger FUSE transport errors on Drive.
    # pyarrow and Spark readers do not need .crc files.
    files = sorted([
        f for f in os.listdir(GOLD_SRC)
        if not f.startswith('.') and (f.endswith('.parquet') or f == '_SUCCESS')
    ])
    print(f'  {len(files)} file(s) to copy.')
    for i, fname in enumerate(files, 1):
        src = os.path.join(GOLD_SRC, fname)
        dst = os.path.join(LOCAL_GOLD, fname)
        shutil.copy2(src, dst)
        # Verify the copy is complete — a size mismatch means a partial FUSE write.
        if fname.endswith('.parquet') and os.path.getsize(dst) != os.path.getsize(src):
            raise RuntimeError(f'Size mismatch after copy: {fname}')
        print(f'  [{i}/{len(files)}] {fname} ✓ ({os.path.getsize(dst)/1e6:.1f} MB)')
    print('Done.')

Copying gold parquet files to local disk ...
  51 file(s) to copy.
  [1/51] _SUCCESS ✓ (0.0 MB)
  [2/51] part-00000-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (62.8 MB)
  [3/51] part-00001-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (65.7 MB)
  [4/51] part-00002-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (60.8 MB)
  [5/51] part-00003-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (61.5 MB)
  [6/51] part-00004-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (60.8 MB)
  [7/51] part-00005-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (61.5 MB)
  [8/51] part-00006-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (62.1 MB)
  [9/51] part-00007-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (60.6 MB)
  [10/51] part-00008-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (62.5 MB)
  [11/51] part-00009-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓ (63.9 MB)
  [12/51] pa

## 5 — Load Gold Data

Read the gold parquet with **pyarrow** directly (no Spark required for inference).  
We load all columns needed for anomaly features plus ID/metadata columns for output.

In [6]:
# Columns to load from parquet (IDs + metadata + all feature columns)
# NOTE: user_star_deviation and biz_star_deviation are intentionally excluded (data leakage)
LOAD_COLS = [
    # identifiers (kept for output only)
    'review_id', 'user_id', 'business_id',
    # review metadata
    'stars', 'text',
    # text-based features
    'char_count', 'word_count', 'avg_word_length', 'exclamation_count',
    'question_count', 'uppercase_ratio', 'sentence_count',
    # engagement features
    'useful', 'funny', 'cool', 'total_votes',
    # temporal features
    'review_year', 'review_month', 'review_dow', 'review_hour', 'is_weekend',
    # user features
    'user_review_count', 'user_avg_stars', 'user_useful_votes', 'user_funny_votes',
    'user_cool_votes', 'user_fans', 'user_is_elite', 'user_elite_years',
    'user_tenure_days', 'user_engagement_score',
    # business features
    'biz_stars', 'biz_review_count', 'biz_is_open', 'biz_category_count',
    'biz_latitude', 'biz_longitude',
    # sentiment
    'sentiment_label', 'sentiment_binary',
]

# Read from local disk (copied above to avoid Drive FUSE timeout)
print('Reading gold parquet from local disk ...')
table = pq.read_table(LOCAL_GOLD, columns=LOAD_COLS)
df    = table.to_pandas()
del table

print(f'Loaded : {len(df):,} rows x {df.shape[1]} columns')
print(f'Memory : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
df.head(3)

Reading gold parquet from local disk ...
Loaded : 6,990,280 rows x 39 columns
Memory : 7.07 GB


,review_id,user_id,business_id,stars,text,char_count,word_count,avg_word_length,exclamation_count,question_count,...,user_tenure_days,user_engagement_score,biz_stars,biz_review_count,biz_is_open,biz_category_count,biz_latitude,biz_longitude,sentiment_label,sentiment_binary
0,gtaE5Mu317tS4QHa0J0hfQ,---UgP94gokyCDuB5zUssA,hKr-RKMVpj3gRkSWcjg3Zw,3,So I was in the mood for some Tacos and not Ta...,558,110,4.081818,1,0,...,4211.0,5.723585,3.0,14,0,3.0,30.004618,-90.241028,1,NaN
1,U_R48DfSdW-Ef0ilabeoUw,---UgP94gokyCDuB5zUssA,GBTPC53ZrG1ZBY3DT8Mbcw,4,Overall Great food with a few minor hiccups. I...,372,67,4.567164,0,0,...,4211.0,5.723585,4.0,4554,1,11.0,29.950741,-90.070419,2,1.0
2,Y-xvJ7cN2Sul8Ub-qypLSg,---UgP94gokyCDuB5zUssA,mnq8JNUjIBwUoLBk-b2V9g,4,I come here at least twice a week and to Start...,369,70,4.285714,1,0,...,4211.0,5.723585,4.0,67,1,4.0,29.950180,-90.074234,2,1.0


In [7]:
# Clean: fill numeric NaNs with 0, drop rows where all feature columns are NaN
FEATURE_COLS_CHECK = [
    'char_count', 'word_count', 'user_review_count', 'biz_stars'
]
before = len(df)
df.dropna(subset=FEATURE_COLS_CHECK, how='all', inplace=True)
df.fillna(0, inplace=True)

print(f'Rows before cleaning : {before:,}')
print(f'Rows after cleaning  : {len(df):,}')
print(f'Dropped              : {before - len(df):,}')
df.reset_index(drop=True, inplace=True)

Rows before cleaning : 6,990,280
Rows after cleaning  : 6,990,280
Dropped              : 0


## 6 — Define Anomaly Features

32 tabular features covering text quality, engagement, temporal patterns, user behaviour, and business context.  
`user_star_deviation` and `biz_star_deviation` are **excluded** to avoid data leakage.

In [8]:
ANOMALY_FEATURES = [
    # Text quality signals
    'char_count', 'word_count', 'avg_word_length', 'exclamation_count',
    'question_count', 'uppercase_ratio', 'sentence_count',
    # Engagement
    'useful', 'funny', 'cool', 'total_votes',
    # Temporal
    'review_year', 'review_month', 'review_dow', 'review_hour', 'is_weekend',
    # User behaviour
    'user_review_count', 'user_avg_stars', 'user_useful_votes', 'user_funny_votes',
    'user_cool_votes', 'user_fans', 'user_is_elite', 'user_elite_years',
    'user_tenure_days', 'user_engagement_score',
    # Business context
    'biz_stars', 'biz_review_count', 'biz_is_open', 'biz_category_count',
    'biz_latitude', 'biz_longitude',
]

print(f'Total anomaly features : {len(ANOMALY_FEATURES)}')
print('Features:')
for i, f in enumerate(ANOMALY_FEATURES, 1):
    print(f'  {i:2d}. {f}')

Total anomaly features : 32
Features:
   1. char_count
   2. word_count
   3. avg_word_length
   4. exclamation_count
   5. question_count
   6. uppercase_ratio
   7. sentence_count
   8. useful
   9. funny
  10. cool
  11. total_votes
  12. review_year
  13. review_month
  14. review_dow
  15. review_hour
  16. is_weekend
  17. user_review_count
  18. user_avg_stars
  19. user_useful_votes
  20. user_funny_votes
  21. user_cool_votes
  22. user_fans
  23. user_is_elite
  24. user_elite_years
  25. user_tenure_days
  26. user_engagement_score
  27. biz_stars
  28. biz_review_count
  29. biz_is_open
  30. biz_category_count
  31. biz_latitude
  32. biz_longitude


## 7 — Scale Features

In [9]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[ANOMALY_FEATURES].values)

print(f'Feature matrix : {X_scaled.shape}')
print(f'Mean (first 5) : {X_scaled[:, :5].mean(axis=0).round(4)}')
print(f'Std  (first 5) : {X_scaled[:, :5].std(axis=0).round(4)}')

Feature matrix : (6990280, 32)
Mean (first 5) : [-0.  0.  0.  0.  0.]
Std  (first 5) : [1. 1. 1. 1. 1.]


---
# Part 1 — Isolation Forest

Isolation Forest detects anomalies by randomly partitioning the feature space.
Anomalous points are isolated in fewer splits (shorter path length from root to leaf).
We set `contamination=0.05` assuming ~5% of reviews may be suspicious.

## 8 — Train Isolation Forest

In [10]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # assume 5% suspicious reviews
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

print('Training Isolation Forest ...')
print(f'  n_estimators : {iso_forest.n_estimators}')
print(f'  contamination: {iso_forest.contamination}')
print(f'  n_jobs       : {iso_forest.n_jobs}')

iso_forest.fit(X_scaled)
print('Training complete.')

Training Isolation Forest ...
  n_estimators : 200
  contamination: 0.05
  n_jobs       : -1
Training complete.


## 9 — Score & Flag Reviews

In [11]:
print('Computing anomaly scores ...')
anomaly_scores = iso_forest.decision_function(X_scaled)  # higher = more normal
iso_labels     = iso_forest.predict(X_scaled)            # -1=anomaly, 1=normal

df['iso_score'] = anomaly_scores
df['iso_flag']  = (iso_labels == -1).astype(int)

n_flagged = df['iso_flag'].sum()
n_normal  = len(df) - n_flagged

print(f'Total reviews           : {len(df):,}')
print(f'Flagged as suspicious   : {n_flagged:,} ({n_flagged/len(df)*100:.2f}%)')
print(f'Classified as normal    : {n_normal:,} ({n_normal/len(df)*100:.2f}%)')

print(f'\nAnomaly score stats:')
print(df['iso_score'].describe().round(4))

Computing anomaly scores ...
Total reviews           : 6,990,280
Flagged as suspicious   : 349,514 (5.00%)
Classified as normal    : 6,640,766 (95.00%)

Anomaly score stats:
count    6.990280e+06
mean     1.022000e-01
std      5.260000e-02
min     -2.692000e-01
25%      8.470000e-02
50%      1.158000e-01
75%      1.369000e-01
max      1.780000e-01
Name: iso_score, dtype: float64


## 10 — Isolation Forest Visualizations

In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Isolation Forest — Anomaly Analysis', fontsize=15, fontweight='bold', y=1.01)

# ── Plot 1: Anomaly score distribution ─────────────────────────────────────
ax = axes[0, 0]
for flag, color, label in [(0, 'steelblue', 'Normal'), (1, 'firebrick', 'Suspicious')]:
    subset = df[df['iso_flag'] == flag]['iso_score']
    ax.hist(subset, bins=80, alpha=0.65, color=color, label=f'{label} (n={len(subset):,})',
            density=True, edgecolor='none')
ax.axvline(0, color='black', linestyle='--', linewidth=1.2, label='Decision boundary')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Density')
ax.set_title('Anomaly Score Distribution')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Plot 2: Anomaly rate by star rating ────────────────────────────────────
ax = axes[0, 1]
star_anomaly = df.groupby('stars')['iso_flag'].agg(['mean', 'count']).reset_index()
star_anomaly['mean_pct'] = star_anomaly['mean'] * 100
bars = ax.bar(star_anomaly['stars'], star_anomaly['mean_pct'],
              color=['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0'],
              alpha=0.85, edgecolor='white', linewidth=0.8)
for bar, pct in zip(bars, star_anomaly['mean_pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Suspicious Rate (%)')
ax.set_title('Anomaly Rate by Star Rating')
ax.set_xticks([1, 2, 3, 4, 5])
ax.grid(axis='y', alpha=0.3)

# ── Plot 3: Anomaly rate by user tenure bucket ─────────────────────────────
ax = axes[1, 0]
bins    = [0, 90, 365, 730, 1825, 3650, df['user_tenure_days'].max() + 1]
labels  = ['<3m', '3m-1y', '1-2y', '2-5y', '5-10y', '10y+']
df['tenure_bucket'] = pd.cut(df['user_tenure_days'], bins=bins, labels=labels, right=False)
tenure_anom = df.groupby('tenure_bucket', observed=True)['iso_flag'].mean() * 100
colors_t = sns.color_palette('coolwarm', n_colors=len(labels))
bars_t = ax.bar(tenure_anom.index.astype(str), tenure_anom.values,
                color=colors_t, alpha=0.85, edgecolor='white')
for bar, val in zip(bars_t, tenure_anom.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xlabel('User Tenure')
ax.set_ylabel('Suspicious Rate (%)')
ax.set_title('Anomaly Rate by User Tenure')
ax.grid(axis='y', alpha=0.3)

# ── Plot 4: PCA scatter (sample 5k) ────────────────────────────────────────
ax = axes[1, 1]
pca        = PCA(n_components=2, random_state=42)
sample_idx = np.random.RandomState(42).choice(len(X_scaled), size=min(5000, len(X_scaled)), replace=False)
X_pca      = pca.fit_transform(X_scaled[sample_idx])
flags_pca  = df['iso_flag'].values[sample_idx]

for flag, color, label, alpha in [(0, 'steelblue', 'Normal', 0.3), (1, 'firebrick', 'Suspicious', 0.7)]:
    mask = flags_pca == flag
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, alpha=alpha, s=8, label=f'{label} (n={mask.sum():,})')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('PCA Scatter — Normal vs Suspicious (5k sample)')
ax.legend(fontsize=9, markerscale=2)
ax.grid(alpha=0.3)

plt.tight_layout()
save_fig('01_isolation_forest_analysis')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/01_isolation_forest_analysis.png


## 11 — Suspicious vs Normal: Feature Comparison

In [13]:
comparison = df.groupby('iso_flag')[ANOMALY_FEATURES].mean().T
comparison.columns = ['Normal', 'Suspicious']
comparison['diff_pct'] = (
    (comparison['Suspicious'] - comparison['Normal'])
    / comparison['Normal'].abs().replace(0, np.nan)
    * 100
).round(1)

top15 = comparison.sort_values('diff_pct', key=abs, ascending=False).head(15)
print('=== Top 15 Features Differentiating Suspicious vs Normal Reviews ===')
print(top15.to_string())

=== Top 15 Features Differentiating Suspicious vs Normal Reviews ===
                           Normal   Suspicious  diff_pct
user_funny_votes        33.858624  2878.630999    8401.9
user_cool_votes         57.151394  4715.470576    8150.8
user_useful_votes      121.963249  6242.459899    5018.3
user_fans                4.443630   179.753011    3945.2
funny                    0.187080     2.976668    1491.1
cool                     0.290019     4.461993    1438.5
user_review_count       79.373602   968.556149    1120.2
total_votes              1.360565    14.344976     954.3
useful                   0.883466     6.906316     681.7
user_elite_years         0.987195     7.363974     645.9
question_count           0.132928     0.837423     530.0
user_is_elite            0.214140     0.868652     305.6
user_engagement_score    7.027736    17.952221     155.4
char_count             527.719713  1328.614184     151.8
word_count              97.442147   244.106911     150.5


In [14]:
# Visualize top feature differences
fig, ax = plt.subplots(figsize=(12, 7))

colors_bar = ['#F44336' if v > 0 else '#2196F3' for v in top15['diff_pct']]
bars = ax.barh(top15.index, top15['diff_pct'], color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)

for bar, val in zip(bars, top15['diff_pct']):
    x_pos = bar.get_width() + (1 if val >= 0 else -1)
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f'{val:+.1f}%', va='center', fontsize=8,
            ha='left' if val >= 0 else 'right')

ax.set_xlabel('% Difference (Suspicious vs Normal)')
ax.set_title('Top Feature Differences: Suspicious vs Normal Reviews', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

import matplotlib.patches as mpatches
red_patch  = mpatches.Patch(color='#F44336', alpha=0.85, label='Higher in Suspicious')
blue_patch = mpatches.Patch(color='#2196F3', alpha=0.85, label='Lower in Suspicious')
ax.legend(handles=[red_patch, blue_patch], fontsize=10)

plt.tight_layout()
save_fig('02_feature_differences')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/02_feature_differences.png


## 12 — Isolation Forest: Feature Importance Proxy

We compute the mean anomaly score per feature quartile to identify which features
drive the most separation between normal and suspicious reviews.

In [15]:
# Compute correlation between each feature and anomaly score (proxy for feature importance)
feature_importance = {}
for feat in ANOMALY_FEATURES:
    corr = np.corrcoef(df[feat].values, df['iso_score'].values)[0, 1]
    feature_importance[feat] = abs(corr)  # absolute correlation

fi_series = pd.Series(feature_importance).sort_values(ascending=False)

print('=== Feature Importance Proxy (|correlation with anomaly score|) ===')
print(fi_series.round(4).to_string())

=== Feature Importance Proxy (|correlation with anomaly score|) ===
user_engagement_score    0.6696
user_elite_years         0.6641
user_review_count        0.5983
user_is_elite            0.5679
total_votes              0.5360
char_count               0.4941
useful                   0.4908
word_count               0.4897
sentence_count           0.4849
user_fans                0.4773
cool                     0.4757
user_useful_votes        0.4573
funny                    0.4089
user_cool_votes          0.4085
user_funny_votes         0.3485
question_count           0.3070
user_tenure_days         0.2769
review_year              0.2074
biz_is_open              0.2007
exclamation_count        0.1623
biz_longitude            0.0954
biz_stars                0.0888
biz_review_count         0.0708
user_avg_stars           0.0707
uppercase_ratio          0.0704
is_weekend               0.0664
review_hour              0.0585
biz_category_count       0.0455
biz_latitude             0.0394
avg_

In [16]:
fig, ax = plt.subplots(figsize=(12, 8))

colors_fi = sns.color_palette('viridis', n_colors=len(fi_series))
bars = ax.barh(fi_series.index, fi_series.values, color=colors_fi, alpha=0.85, edgecolor='white')

ax.set_xlabel('|Correlation with Isolation Forest Anomaly Score|')
ax.set_title('Isolation Forest — Feature Importance Proxy', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
save_fig('02b_feature_importance_proxy')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/02b_feature_importance_proxy.png


---
# Part 2 — Autoencoder

An Autoencoder learns to compress and reconstruct normal data patterns.
Reviews that deviate from normal patterns will have **high reconstruction error**,
making them candidates for anomaly flagging.

Architecture: **Input(32) → Dense(64) → BN → Dropout → Dense(32) → BN → Bottleneck(8) → Dense(32) → BN → Dense(64) → BN → Output(32)**

## 13 — Autoencoder Architecture

In [17]:
INPUT_DIM    = len(ANOMALY_FEATURES)
ENCODING_DIM = 8

inp = Input(shape=(INPUT_DIM,), name='input')

# ── Encoder ────────────────────────────────────────────────────────────────
x       = layers.Dense(64, activation='relu', name='enc_dense_1')(inp)
x       = layers.BatchNormalization(name='enc_bn_1')(x)
x       = layers.Dropout(0.2, name='enc_drop_1')(x)
x       = layers.Dense(32, activation='relu', name='enc_dense_2')(x)
x       = layers.BatchNormalization(name='enc_bn_2')(x)
encoded = layers.Dense(ENCODING_DIM, activation='relu', name='bottleneck')(x)

# ── Decoder ────────────────────────────────────────────────────────────────
x       = layers.Dense(32, activation='relu', name='dec_dense_1')(encoded)
x       = layers.BatchNormalization(name='dec_bn_1')(x)
x       = layers.Dense(64, activation='relu', name='dec_dense_2')(x)
x       = layers.BatchNormalization(name='dec_bn_2')(x)
decoded = layers.Dense(INPUT_DIM, activation='linear', name='output')(x)

autoencoder = Model(inputs=inp, outputs=decoded, name='anomaly_autoencoder')
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse'
)
autoencoder.summary()

print(f'\nInput dim    : {INPUT_DIM}')
print(f'Bottleneck   : {ENCODING_DIM}')
print(f'Compression  : {INPUT_DIM / ENCODING_DIM:.1f}x')

Model: "anomaly_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_dense_1 (Dense)             │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_bn_1 (BatchNormalization)   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_drop_1 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_dense_2 (Dense)             │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_bn_2 (BatchNormalization)   │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 8)              │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_dense_1 (Dense)             │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_bn_1 (BatchNormalization)   │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_dense_2 (Dense)             │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_bn_2 (BatchNormalization)   │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 32)             │         2,080 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,704 (37.91 KB)

 Trainable params: 9,320 (36.41 KB)

 Non-trainable params: 384 (1.50 KB)


Input dim    : 32
Bottleneck   : 8
Compression  : 4.0x


## 14 — Train Autoencoder

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print('Training Autoencoder ...')
print(f'  Training on {X_scaled.shape[0]:,} samples')
print(f'  Epochs         : 30 (with early stopping, patience=5)')
print(f'  Batch size     : 2048')
print(f'  Validation     : 10%')

ae_history = autoencoder.fit(
    X_scaled, X_scaled,         # input = target (reconstruction)
    epochs=30,
    batch_size=2048,
    validation_split=0.1,
    callbacks=[early_stop],
    shuffle=True,
    verbose=1
)

print(f'\nTraining stopped at epoch {len(ae_history.history["loss"])}')
print(f'Best val_loss : {min(ae_history.history["val_loss"]):.6f}')

Training Autoencoder ...
  Training on 6,990,280 samples
  Epochs         : 30 (with early stopping, patience=5)
  Batch size     : 2048
  Validation     : 10%
Epoch 1/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - loss: 0.3557 - val_loss: 0.2178
Epoch 2/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.2232 - val_loss: 0.1651
Epoch 3/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1870 - val_loss: 0.1475
Epoch 4/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1733 - val_loss: 0.1473
Epoch 5/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1669 - val_loss: 0.1457
Epoch 6/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1625 - val_loss: 0.1423
Epoch 7/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1567 - val_loss: 0.1320
Epoch 8/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1537 - val_loss: 0.1271
Epoch 9/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.1519 - val_loss: 0.1720
Epoch 10/30
3072/3072 ━━━━━━━━━━━━━━━━━━━━ 8s 3

## 15 — Autoencoder Training Curves

In [19]:
hist    = ae_history.history
epochs  = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Autoencoder Training — Reconstruction Loss (MSE)', fontsize=13, fontweight='bold')

# ── Full loss curve ─────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs, hist['loss'],     'b-o', linewidth=2, markersize=4, label='Train Loss')
ax.plot(epochs, hist['val_loss'], 'r-o', linewidth=2, markersize=4, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Train vs Validation Loss')
ax.legend()
ax.grid(alpha=0.3)

# ── Log-scale loss curve ────────────────────────────────────────────────────
ax = axes[1]
ax.semilogy(epochs, hist['loss'],     'b-o', linewidth=2, markersize=4, label='Train Loss')
ax.semilogy(epochs, hist['val_loss'], 'r-o', linewidth=2, markersize=4, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (log scale)')
ax.set_title('Train vs Validation Loss (Log Scale)')
ax.legend()
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_fig('03_autoencoder_training_curve')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/03_autoencoder_training_curve.png


## 16 — Reconstruction Error & Threshold

In [20]:
print('Computing reconstruction errors ...')
X_reconstructed = autoencoder.predict(X_scaled, batch_size=2048, verbose=1)
recon_error     = np.mean(np.square(X_scaled - X_reconstructed), axis=1)

df['recon_error'] = recon_error

# Threshold: top 5% = anomalies
threshold  = np.percentile(recon_error, 95)
df['ae_flag'] = (recon_error > threshold).astype(int)

print(f'\nReconstruction error statistics:')
print(pd.Series(recon_error).describe().round(6).to_string())
print(f'\nThreshold (95th percentile) : {threshold:.6f}')
print(f'AE flagged                  : {df["ae_flag"].sum():,} ({df["ae_flag"].mean()*100:.2f}%)')

Computing reconstruction errors ...
3414/3414 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step

Reconstruction error statistics:
count    6.990280e+06
mean     1.129810e-01
std      1.038162e+00
min      3.436000e-03
25%      4.661600e-02
50%      7.369000e-02
75%      1.241280e-01
max      1.028033e+03

Threshold (95th percentile) : 0.288284
AE flagged                  : 349,514 (5.00%)


## 17 — Autoencoder Visualizations

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Autoencoder — Reconstruction Error Analysis', fontsize=14, fontweight='bold')

# ── Plot 1: Reconstruction error distribution ───────────────────────────────
ax = axes[0]
# Clip to 99th percentile for readability
clip_val = np.percentile(recon_error, 99)
clipped  = np.clip(recon_error, 0, clip_val)

ax.hist(clipped[df['ae_flag'] == 0], bins=100, alpha=0.65,
        color='steelblue', density=True, label='Normal', edgecolor='none')
ax.hist(clipped[df['ae_flag'] == 1], bins=100, alpha=0.65,
        color='firebrick', density=True, label='Anomaly (AE)', edgecolor='none')
ax.axvline(threshold, color='black', linestyle='--', linewidth=1.5,
           label=f'Threshold = {threshold:.4f}')
ax.set_xlabel('Reconstruction Error (MSE, clipped at 99th pct)')
ax.set_ylabel('Density')
ax.set_title('Reconstruction Error Distribution')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Plot 2: Reconstruction error by star rating (box plot) ──────────────────
ax = axes[1]
star_groups = [df[df['stars'] == s]['recon_error'].values for s in [1, 2, 3, 4, 5]]
# Clip for readability
star_groups_clipped = [np.clip(g, 0, clip_val) for g in star_groups]

bp = ax.boxplot(
    star_groups_clipped,
    labels=['1 star', '2 stars', '3 stars', '4 stars', '5 stars'],
    patch_artist=True,
    showfliers=False,
    medianprops={'color': 'black', 'linewidth': 2}
)
colors_bp = ['#F44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.axhline(threshold, color='black', linestyle='--', linewidth=1.2,
           label=f'Threshold = {threshold:.4f}')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Reconstruction Error (clipped at 99th pct)')
ax.set_title('Reconstruction Error by Star Rating')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_fig('03_autoencoder_analysis')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/03_autoencoder_analysis.png


In [22]:
# Additional: AE anomaly rate by star rating
ae_star = df.groupby('stars')['ae_flag'].mean() * 100
print('=== AE Anomaly Rate by Star Rating ===')
for star, rate in ae_star.items():
    print(f'  {star} stars : {rate:.2f}%')

print(f'\nMedian recon_error (normal)    : {df[df["ae_flag"]==0]["recon_error"].median():.6f}')
print(f'Median recon_error (anomaly)   : {df[df["ae_flag"]==1]["recon_error"].median():.6f}')

=== AE Anomaly Rate by Star Rating ===
  1 stars : 5.80%
  2 stars : 5.10%
  3 stars : 6.40%
  4 stars : 6.28%
  5 stars : 3.84%

Median recon_error (normal)    : 0.070507
Median recon_error (anomaly)   : 0.388188


---
# Part 3 — Model Comparison & Combined Analysis

## 18 — Agreement Between Models

In [23]:
# Reviews flagged by BOTH models = high-confidence suspicious
df['both_flagged'] = ((df['iso_flag'] == 1) & (df['ae_flag'] == 1)).astype(int)

only_iso  = ((df['iso_flag'] == 1) & (df['ae_flag'] == 0)).sum()
only_ae   = ((df['iso_flag'] == 0) & (df['ae_flag'] == 1)).sum()
both      = df['both_flagged'].sum()
neither   = ((df['iso_flag'] == 0) & (df['ae_flag'] == 0)).sum()
total     = len(df)

print('=== Model Agreement Summary ===')
print(f'  Total reviews          : {total:,}')
print(f'  Only Isolation Forest  : {only_iso:,} ({only_iso/total*100:.2f}%)')
print(f'  Only Autoencoder       : {only_ae:,} ({only_ae/total*100:.2f}%)')
print(f'  Flagged by BOTH        : {both:,} ({both/total*100:.2f}%)  <-- high-confidence suspicious')
print(f'  Flagged by NEITHER     : {neither:,} ({neither/total*100:.2f}%)')

# Agreement rate
agreement = (both + neither) / total * 100
print(f'\n  Overall agreement      : {agreement:.2f}%')

=== Model Agreement Summary ===
  Total reviews          : 6,990,280
  Only Isolation Forest  : 175,904 (2.52%)
  Only Autoencoder       : 175,904 (2.52%)
  Flagged by BOTH        : 173,610 (2.48%)  <-- high-confidence suspicious
  Flagged by NEITHER     : 6,464,862 (92.48%)

  Overall agreement      : 94.97%


## 19 — Agreement Bar Chart

In [24]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Isolation Forest vs Autoencoder — Model Agreement', fontsize=14, fontweight='bold')

# ── Plot 1: Agreement breakdown bar chart ───────────────────────────────────
ax = axes[0]
categories = ['Only IF\n(iso_flag only)', 'Only AE\n(ae_flag only)',
               'Both Models\n(high-confidence)', 'Neither\n(normal)']
counts     = [only_iso, only_ae, both, neither]
pcts       = [c / total * 100 for c in counts]
colors_ag  = ['#FF9800', '#9C27B0', '#F44336', '#4CAF50']

bars = ax.bar(categories, pcts, color=colors_ag, alpha=0.85, edgecolor='white', linewidth=0.8)
for bar, count, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{pct:.2f}%\n({count:,})', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Percentage of Reviews')
ax.set_title('Flagging Breakdown by Model Agreement')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(pcts) * 1.25)

# ── Plot 2: Scatter — iso_score vs recon_error (sample 10k) ────────────────
ax = axes[1]
sample_idx2 = np.random.RandomState(123).choice(len(df), size=min(10000, len(df)), replace=False)
sample_df   = df.iloc[sample_idx2]

group_map   = {
    (0, 0): ('steelblue', 0.2, 'Neither'),
    (1, 0): ('orange',    0.6, 'Only IF'),
    (0, 1): ('purple',    0.6, 'Only AE'),
    (1, 1): ('firebrick', 0.9, 'Both (high-conf.)'),
}
for (if_flag, ae_flag_val), (color, alpha, label) in group_map.items():
    mask = (sample_df['iso_flag'] == if_flag) & (sample_df['ae_flag'] == ae_flag_val)
    ax.scatter(
        sample_df.loc[mask, 'iso_score'],
        np.clip(sample_df.loc[mask, 'recon_error'], 0, np.percentile(recon_error, 99)),
        c=color, alpha=alpha, s=10, label=f'{label} (n={mask.sum():,})'
    )

ax.axvline(0, color='black', linestyle='--', linewidth=0.9, label='IF boundary')
ax.axhline(threshold, color='gray', linestyle='--', linewidth=0.9, label=f'AE threshold ({threshold:.3f})')
ax.set_xlabel('Isolation Forest Score (lower = more anomalous)')
ax.set_ylabel('Reconstruction Error (clipped at 99th pct)')
ax.set_title('IF Score vs AE Reconstruction Error (10k sample)')
ax.legend(fontsize=8, markerscale=2)
ax.grid(alpha=0.3)

plt.tight_layout()
save_fig('04_model_agreement')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/04_model_agreement.png


## 20 — Top Suspicious Reviews Analysis

In [25]:
# Combine scores: normalise iso_score (invert so higher = more suspicious) and recon_error
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler()
iso_norm    = mms.fit_transform(-df['iso_score'].values.reshape(-1, 1)).flatten()  # invert: lower score = more suspicious
recon_norm  = mms.fit_transform(df['recon_error'].values.reshape(-1, 1)).flatten()

df['combined_suspicion'] = (iso_norm + recon_norm) / 2

# Top 10 most suspicious among those flagged by both models
high_conf = df[df['both_flagged'] == 1].copy()
top10 = high_conf.nlargest(10, 'combined_suspicion')[
    ['review_id', 'user_id', 'business_id', 'stars', 'text',
     'iso_score', 'recon_error', 'combined_suspicion',
     'user_review_count', 'user_tenure_days', 'char_count', 'word_count']
].copy()

top10['text_snippet'] = top10['text'].astype(str).str[:120] + '...'

print('=== Top 10 Most Suspicious Reviews (flagged by both models) ===')
display_cols = ['review_id', 'stars', 'iso_score', 'recon_error', 'combined_suspicion',
                'user_review_count', 'user_tenure_days', 'char_count', 'text_snippet']
pd.set_option('display.max_colwidth', 130)
display(top10[display_cols].reset_index(drop=True))

=== Top 10 Most Suspicious Reviews (flagged by both models) ===


,review_id,stars,iso_score,recon_error,combined_suspicion,user_review_count,user_tenure_days,char_count,text_snippet
0,pL_jmXOSPOMq71ZwrlRynQ,1,-0.128832,860.761115,0.761748,2.0,5683.0,2100,Went there for a birthday dinner and had reservations for 9 people (and 3 additional people showed up) after a grueling ...
1,_qVk19I56FN8F-3xynu9wQ,2,-0.087068,617.344396,0.596662,390.0,5075.0,2680,I was so excited to try this place on our recent Nashville visit. PPPPFFFFTTTTTZZZZZ. That's the sound of the air defl...
2,nsbGQZ19xDoGF-DQRWx6Wg,5,-0.268421,26.495856,0.512064,17473.0,6197.0,3792,"Omg, one guess where I'm HEADING\nTo the Terminal Market known as READING\nI feel like Starlight when she's SHEDDING\nI fee..."
3,gIeTCLYTr01wsEQrmbRsqg,4,-0.266600,20.451416,0.507088,17473.0,6197.0,3236,"Time for another epic about Bubble Tea? Rhetorical question? It's in Cherry Hill, and as they say, you can take the Cher..."
4,p5HLWOPgp4xib_jOc5IFEQ,5,-0.264230,20.044563,0.504241,17473.0,6197.0,3978,"I'm not into farmer's cheese, I prefer the QUARK WAY\nIf you like it with the lights off, then you prefer the DARKWAY\nIf ..."
5,MEvRDQYgnVj4TrvspZ6D2A,5,-0.269155,2.795352,0.501358,3004.0,5790.0,2023,"""Why YES! They Serve Coffee Straight From The Bowels Of HELL!""\nhttps://www.yelp.com/biz_photos/chapel-tavern-reno?selec..."
6,2SbRDRtCeGbg-8fS6Y_DFg,5,-0.268480,3.764565,0.501074,3004.0,5790.0,2325,"Very Elegant Chill Spot To Hide From The Crowds\n\nSITREP\n\nEverywhere you walk in The Peppermill, there's always a nice su..."
7,nHCIXf2GGgdsNtU49OoJCg,5,-0.268129,2.666730,0.500148,3004.0,5790.0,3534,"Don't Let ""Magic Bullet Hangover Cure"" Drive Your Entertainment Choices! (He's A Mischievous Lil Guy!)\n\nSITREP\n\n...becau..."
8,eosveIrY2BWh616TUbrBnw,5,-0.259661,21.351684,0.499767,17473.0,6197.0,4855,"If someone creepy hits on you when you're in here, it's Pervy Marte.\n\nNow we're going to let Bono introduce this one. (V..."
9,vKpvIAMObPXbTxxvxvXMIQ,3,-0.266008,5.540013,0.499174,1311.0,5848.0,4436,"11/27/17\n\nWe actually saved what we thought would be one of the best meals for very last. On the very last day, last ho..."


In [26]:
# Profile: suspicious vs normal — extended feature comparison
print('=== Statistical Profile: High-Confidence Suspicious vs Normal ===')
profile_cols = [
    'char_count', 'word_count', 'uppercase_ratio', 'exclamation_count',
    'user_review_count', 'user_tenure_days', 'user_fans',
    'user_engagement_score', 'total_votes', 'stars'
]
profile = df.groupby('both_flagged')[profile_cols].agg(['mean', 'median']).round(2)
profile.index = ['Normal', 'High-Conf. Suspicious']
display(profile)

=== Statistical Profile: High-Confidence Suspicious vs Normal ===


char_count         word_count        uppercase_ratio  \
                            mean  median       mean median            mean   
Normal                    546.91   397.0     100.98   73.0            0.03   
High-Conf. Suspicious    1386.60  1100.0     253.91  201.0            0.03   

                             exclamation_count        user_review_count  \
                      median              mean median              mean   
Normal                  0.02              1.29    0.0             97.17   
High-Conf. Suspicious   0.03              2.76    1.0           1170.83   

                             user_tenure_days         user_fans         \
                      median             mean  median      mean median   
Normal                  23.0          4569.19  4611.0      6.80    0.0   
High-Conf. Suspicious  754.0          5522.16  5622.0    264.82  106.0   

                      user_engagement_score        total_votes        stars  \
                                       mean median        mean median  mean   
Normal                             7.290000   6.33        1.57    0.0  3.75   
High-Conf. Suspicious             18.530001  19.58       19.46   12.0  3.69   

                              
                      median  
Normal                   4.0  
High-Conf. Suspicious    4.0

In [27]:
# Heatmap: anomaly rate by stars x review length bucket
df['length_bucket'] = pd.cut(
    df['word_count'],
    bins=[0, 20, 50, 100, 200, 500, df['word_count'].max() + 1],
    labels=['<20', '20-50', '50-100', '100-200', '200-500', '500+']
)

heatmap_data = df.pivot_table(
    index='stars',
    columns='length_bucket',
    values='both_flagged',
    aggfunc='mean',
    observed=True
) * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.1f', cmap='YlOrRd',
    ax=ax, cbar_kws={'label': 'Suspicious Rate (%)'},
    linewidths=0.5
)
ax.set_title('Suspicious Rate (%) by Stars x Review Length',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Review Word Count Bucket')
ax.set_ylabel('Star Rating')
plt.tight_layout()
save_fig('05_suspicion_heatmap_stars_length')

  Saved -> /content/drive/MyDrive/Dataset/results/anomaly_plots/05_suspicion_heatmap_stars_length.png


## 21 — Save Models & Results

In [28]:
import json

# ── Save models ─────────────────────────────────────────────────────────────
joblib.dump(iso_forest, f'{MODELS_DIR}/isolation_forest.pkl')
print('Isolation Forest saved.')

joblib.dump(scaler, f'{MODELS_DIR}/anomaly_scaler.pkl')
print('Scaler saved.')

autoencoder.save(f'{MODELS_DIR}/autoencoder_model.h5')
print('Autoencoder saved.')

# ── Save flagged reviews (both models agree) ─────────────────────────────────
flagged = df[df['both_flagged'] == 1][[
    'review_id', 'user_id', 'business_id', 'stars',
    'iso_score', 'recon_error', 'combined_suspicion',
    'iso_flag', 'ae_flag', 'both_flagged'
]].copy()
flagged_path = f'{RESULTS_DIR}/suspicious_reviews.parquet'
flagged.to_parquet(flagged_path, index=False)
print(f'Suspicious reviews saved -> {flagged_path}  ({len(flagged):,} rows)')

# ── Save all anomaly scores ──────────────────────────────────────────────────
scores_path = f'{RESULTS_DIR}/anomaly_scores.parquet'
df[['review_id', 'iso_score', 'iso_flag', 'recon_error', 'ae_flag',
    'both_flagged', 'combined_suspicion']].to_parquet(scores_path, index=False)
print(f'Anomaly scores saved   -> {scores_path}  ({len(df):,} rows)')

# ── Save summary metrics JSON ────────────────────────────────────────────────
summary = {
    'total_reviews':        int(len(df)),
    'isolation_forest': {
        'n_estimators':     200,
        'contamination':    0.05,
        'flagged':          int(df['iso_flag'].sum()),
        'flagged_pct':      round(df['iso_flag'].mean() * 100, 2),
    },
    'autoencoder': {
        'input_dim':        INPUT_DIM,
        'encoding_dim':     ENCODING_DIM,
        'threshold_95pct':  round(float(threshold), 6),
        'flagged':          int(df['ae_flag'].sum()),
        'flagged_pct':      round(df['ae_flag'].mean() * 100, 2),
        'epochs_trained':   len(ae_history.history['loss']),
        'best_val_loss':    round(float(min(ae_history.history['val_loss'])), 6),
    },
    'combined': {
        'both_flagged':     int(df['both_flagged'].sum()),
        'both_flagged_pct': round(df['both_flagged'].mean() * 100, 2),
        'only_iso':         int(only_iso),
        'only_ae':          int(only_ae),
        'neither':          int(neither),
    },
    'features_used':        ANOMALY_FEATURES,
    'features_excluded':    ['user_star_deviation', 'biz_star_deviation'],
}

summary_path = f'{RESULTS_DIR}/anomaly_detection_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary JSON saved     -> {summary_path}')

print(f'\n--- All results saved. Total suspicious (both): {len(flagged):,} ---')

Isolation Forest saved.
Scaler saved.
Autoencoder saved.
Suspicious reviews saved -> /content/drive/MyDrive/Dataset/results/suspicious_reviews.parquet  (173,610 rows)
Anomaly scores saved   -> /content/drive/MyDrive/Dataset/results/anomaly_scores.parquet  (6,990,280 rows)
Summary JSON saved     -> /content/drive/MyDrive/Dataset/results/anomaly_detection_summary.json

--- All results saved. Total suspicious (both): 173,610 ---
